In [ ]:
!pip install langchain-text-splitters langchain-experimental


In [ ]:
!pip install langchain langgraph langchain-groq selenium beautifulsoup4 langchain-community langchain-huggingface chromadb sentence-transformers fastmcp langchain-mcp-adapters

In [ ]:
!pip install -q langchain-experimental

In [27]:
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "api_key"

In [28]:
import sys
!apt-get update
!apt-get install -y wget unzip
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -fy install
!apt-get install -y chromium-chromedriver

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.
'dpkg' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [29]:
from pathlib import Path
from langgraph.graph import StateGraph, END
from typing import Any, TypedDict
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from selenium import webdriver
from bs4 import BeautifulSoup

In [30]:
def scrape_website(url):
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    html = driver.page_source
    driver.quit()

    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text

In [31]:
# scrape_website("https://beautiful-soup-4.readthedocs.io/en/latest/#quick-start")

In [32]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)
def explain_content(text):
    prompt = f"""
    Explain the following website content in simple terms:

    {text}
    """
    return llm.invoke(prompt).content

In [33]:
!pip install sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1931.17it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [35]:
def rerank(query, docs_with_scores, top_k=5):
    docs = [doc for doc, _ in docs_with_scores]

    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    reranked = list(zip(docs, scores))
    reranked.sort(key=lambda x: x[1], reverse=True)

    return reranked[:top_k]

In [36]:
VECTOR_DB_DIR = (Path.cwd() / "vector_db")
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma(
    collection_name="rag_docs",
    embedding_function=embedding_model,
    persist_directory=str(VECTOR_DB_DIR),
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"Loaded RAG vector store from {VECTOR_DB_DIR}")

@tool
def scrape(url):
    """Scrape a webpage and return its content."""
    return scrape_website(url)

@tool
def explain(text):
    """Explain given text content."""
    return explain_content(text)

RAG_SCORE_THRESHOLD = 0.8

@tool
def rag(query: str, url_filter: str | None = None):
    """Answer a question using the local RAG vector store with reranking."""
    search_kwargs = {"k": 20}
    if url_filter:
        search_kwargs["filter"] = {"url": url_filter}

    matches = vectorstore.similarity_search_with_score(query, **search_kwargs)
    if not matches:
        return "I don't know based on the indexed data."

    relevant_matches = [
        (doc, score) for doc, score in matches if score <= RAG_SCORE_THRESHOLD
    ]
    if not relevant_matches:
        return "I don't know based on the indexed data."
    reranked = rerank(query, relevant_matches, top_k=5)


    context_lines = []
    for rank, (doc, score) in enumerate(relevant_matches, start=1):
        source_url = doc.metadata.get("url", "unknown")
        context_lines.append(f"Source {rank} | url: {source_url} | score: {score:.4f}\n{doc.page_content}")

    context_text = "\n\n".join(context_lines)
    prompt = f"""
    You are a strict RAG assistant.

    Answer ONLY using the context below.
    If the answer is not clearly present, say:
    "I don't know based on the indexed data."

    Question:
    {query}

    Context:
    {context_text}

    Answer:
    """
    return llm.invoke(prompt).content

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1335.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded RAG vector store from c:\Users\light\OneDrive\Desktop\intern\AI_agent\vector_db


In [37]:
class AgentState(TypedDict):
    tools: str
    inputs: str
    last_task:str
    web_url: str
    scraped_content: str
    last_response: str
    task: str
    history: list
    memory_window: int
    mcp_servers: dict
    mcp_server_url: str

In [38]:
TOOLS = """
You have access to the following tools:

1. scrape(url: str)
   - Use this to extract content from a website URL.

2. explain(text: str)
   - Use this to explain or summarize given content.

3. rag(query: str)
   - Use this for factual questions about indexed pages or stored knowledge.
   - Prefer this over explain when the user is asking a question about a scraped page.
   - It should answer only from retrieved chunks in the vector database.

4. mcp_agent(request: str, server_url: str | None = None)
   - Use this to connect to any MCP server and let a LangChain agent use that server's tools.
   - Pass a server config for external servers, or use the local default at http://127.0.0.1:5000/mcp.


Decide which tool to use based on the user input.
If no tool is needed, return: finish
"""

In [39]:
from langchain_experimental.text_splitter import SemanticChunker

window_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)
semantic_splitter = SemanticChunker(embedding_model)

def scrape_node(state: AgentState):
  content = scrape.invoke({"url": state["web_url"]})
  window_chunks = window_splitter.split_text(content)

  docs = []
  for chunk in window_chunks:
      sem_docs = semantic_splitter.create_documents([chunk])
      docs.extend(sem_docs)



  for doc in docs:
      doc.metadata = {"url": state["web_url"], "source": "web_scrape"}

  vectorstore.delete(where={"url": state["web_url"]})
  vectorstore.add_documents(docs)

  return {
      "scraped_content": content,
      "last_response": f"Sliding window + semantic chunking → {len(docs)} chunks."
  }

In [40]:
def explain_node(state: AgentState):
    explanation = explain.invoke({"text": state["scraped_content"]})
    print(explanation)
    return {
        "last_response": explanation
    }

In [41]:
def rag_node(state: AgentState):
    answer = rag.invoke({"query": state["inputs"], "url_filter": state.get("web_url")})
    print(answer)
    return {
        "last_response": answer
    }

In [42]:
def rule_based_task(state: AgentState) -> str:
    user_text = state["inputs"].strip().lower()
    has_scraped_content = bool(state.get("scraped_content"))
    has_url_context = bool(state.get("web_url"))
    last_task = state.get("last_task", "")

    asks_history = any(phrase in user_text for phrase in ["previous", "earlier", "last response", "what did you say", "from history", "before"])
    asks_scrape = any(phrase in user_text for phrase in ["scrape", "fetch", "load", "index", "crawl"])
    asks_explain = any(phrase in user_text for phrase in ["explain", "summarize", "summary", "describe"])
    asks_mcp = any(phrase in user_text for phrase in ["mcp", "tool server", "server tool", "external tool"])

    question_starts = ("what", "why", "how", "when", "where", "who", "which", "compare", "difference")
    is_question = "?" in user_text or user_text.startswith(question_starts)

    if asks_history:
        return "recall"
    if asks_mcp:
        return "mcp_agent"
    if asks_explain and last_task == "scrape" and has_scraped_content:
        return "explain"
    if asks_scrape and last_task != "scrape":
        return "scrape"
    if is_question and (has_scraped_content or has_url_context):
        return "rag"
    if asks_explain and has_scraped_content:
        return "explain"
    return ""

def brain(state: AgentState):
    rule_choice = rule_based_task(state)
    if rule_choice:
        return {"task": rule_choice, "last_task": rule_choice}

    formatted_history = "\n".join([f"User: {t['user']} | Agent: {t['response']}" for t in state['history']])
    prompt = f"""
    You are an agent. Decisions: scrape, explain, rag, recall, mcp_agent, finish.
    History: {formatted_history}
    Input: {state['inputs']}
    Last Task: {state['last_task']}
    Return one word only.
    """
    raw_decision = llm.invoke(prompt).content.strip().lower()
    decision = raw_decision.split()[0] if raw_decision else "finish"
    if decision == "mcp":
        decision = "mcp_agent"

    if decision not in {"scrape", "explain", "rag", "recall", "mcp_agent", "finish"}:
        decision = "finish"

    return {"task": decision, "last_task": decision}

In [43]:
def recall_node(state: AgentState):
    formatted_history = []
    for turn in state['history']:
        formatted_history.append(
            f"User: {turn.get('user', '')} | Task: {turn.get('task', '')} | Agent: {turn.get('response', '')}"
        )
    formatted_history_text = "\n".join(formatted_history)

    prompt = f"""
You answer questions using the conversation memory below.

Conversation memory:
{formatted_history_text}

Current user question:
{state['inputs']}

Answer only from the stored memory. If the answer is not present, say you do not have that earlier response in memory.
"""

    answer = llm.invoke(prompt).content.strip()
    print(answer)
    return {
        "last_response": answer
    }

In [44]:
def route_task(state: AgentState):
    print(state["task"], " called \n")
    return state["task"]

import asyncio
from concurrent.futures import ThreadPoolExecutor
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent

async def invoke_mcp_langchain_agent(request: str, server_url: str | None = None):
    client = MultiServerMCPClient({
        "local_scrapper_rag": {
            "url": server_url or "http://127.0.0.1:5000/mcp",
            "transport": "http",
        }
    })
    tools = await client.get_tools()
    if not tools:
        return "No tools were discovered from the configured MCP server."
    print(f"Discovered MCP tools: {[tool.name for tool in tools]}")
    
    tool_names = ", ".join(sorted(tool.name for tool in tools))
    system_prompt = f"""
You are an MCP tool-use agent running fully inside LangChain/LangGraph.
Use the connected MCP tools when they help answer the user request.
Available MCP tools: {tool_names}
"""
    mcp_tool_agent = create_agent(llm, tools)
    result = await mcp_tool_agent.ainvoke({
        "messages": [("system", system_prompt), ("human", request)]
    })
    print(f"MCP agent raw result: {result}")
    messages = result.get("messages", [])
    if not messages:
        return str(result)
    content = messages[-1].content
    return content if isinstance(content, str) else str(content)

def run_async_in_worker(coro):
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(lambda: asyncio.run(coro)).result()

@tool("mcp_agent")
def mcp_agent(request: str, server_url: str | None = None) -> str:
    """Connect to an MCP server and use its tools through a LangChain agent."""
    try:
        return run_async_in_worker(invoke_mcp_langchain_agent(request, server_url))
    except Exception as exc:
        return f"MCP agent failed: {exc}"

def mcp_agent_node(state: AgentState):
    try:
        request = state["inputs"]
        if state.get("web_url"):
            request = f"{request}\nURL: {state['web_url']}"
        answer = run_async_in_worker(
            invoke_mcp_langchain_agent(request, state.get("mcp_server_url") or None)
        )
    except Exception as exc:
        answer = f"MCP agent failed: {exc}"
    print(answer)
    return {"last_response": answer}

In [45]:
builder = StateGraph(AgentState)

builder.add_node("brain", brain)
builder.add_node("scrape", scrape_node)
builder.add_node("explain", explain_node)
builder.add_node("rag", rag_node)
builder.add_node("recall", recall_node)
builder.add_node("mcp_agent", mcp_agent_node)

builder.set_entry_point("brain")

builder.add_conditional_edges(
    "brain",
    route_task,
    {
        "scrape": "scrape",
        "explain": "explain",
        "rag": "rag",
        "recall": "recall",
        "mcp_agent": "mcp_agent",
        "finish": END,
    },
)

builder.add_edge("scrape", "brain")
builder.add_edge("explain", END)
builder.add_edge("rag", END)
builder.add_edge("recall", END)
builder.add_edge("mcp_agent", END)

graph = builder.compile()

In [46]:
state = {
    "inputs": "",
    "tools": TOOLS,
    "web_url": "https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant",
    "scraped_content": "",
    "last_response": "",
    "task": "",
    "history": [],
    "last_task": "",
    "memory_window": 5,
    "mcp_servers": {},
    "mcp_server_url": ""
}

def run_agent(user_text, web_url = None, mcp_servers = None, mcp_server_url = None):
    global state

    state["inputs"] = user_text
    if web_url is not None:
        state["web_url"] = web_url
    if mcp_servers is not None:
        state["mcp_servers"] = mcp_servers
    if mcp_server_url is not None:
        state["mcp_server_url"] = mcp_server_url

    result = graph.invoke(state)
    state.update(result)

    executed_task = state.get("task", "finish")
    state["last_task"] = executed_task
    state["history"].append({
        "user": user_text,
        "task": executed_task,
        "response": state.get("last_response", ""),
    })

    window = state.get("memory_window", 5)
    state["history"] = state["history"][-window:]

    return state

In [47]:
# Notebook MCP client check only.
# Start the server separately in PowerShell with: python .\mcp_scrapper_server.py
MCP_SERVER_URL = "http://127.0.0.1:5000/mcp"

async def check_mcp_connection(server_url: str = MCP_SERVER_URL):
    client = MultiServerMCPClient({
        "scrapper_rag": {
            "transport": "http",
            "url": server_url,
        }
    })
    tools = await client.get_tools()
    print("MCP tools:", [tool.name for tool in tools])
    return tools

print(f"Notebook will connect to MCP server at {MCP_SERVER_URL}")


Notebook will connect to MCP server at http://127.0.0.1:5000/mcp


In [48]:
# Run this cell after starting mcp_scrapper_server.py in a terminal.
await check_mcp_connection()


MCP tools: ['scrape', 'explain', 'rag']


[StructuredTool(name='scrape', description='Scrape a webpage URL and return visible text content.', args_schema={'additionalProperties': False, 'properties': {'url': {'type': 'string'}}, 'required': ['url'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000019104CA6D40>),
 StructuredTool(name='explain', description='Explain or summarize the provided text in simple terms.', args_schema={'additionalProperties': False, 'properties': {'text': {'type': 'string'}}, 'required': ['text'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x0000019104C60A60>),
 StructuredTool(name='rag', description='Answer a question using the local vector_db knowledge base.', args_schema={'additionalProperties': False, 'properties': {'query'

In [ ]:
run_agent("using mcp agents scrape this website ","https://en.wikipedia.org/wiki/Main_Page" )


mcp_agent  called 



In [ ]:
run_agent("explain this website")

In [ ]:
run_agent("what is Wikipedia?")